## simple 1D spring based contact
two rigid bodies, target_rb, and end_effector_rb

In [2]:
import torch
print(torch.cuda.is_available())

True


In [5]:
#robot hardware
AF_FOIL_LENGTH = 0.8128 #32" blade to Meters

In [48]:
import sympy as sp

##Rigidbody dynamics
from sympy.physics.mechanics import dynamicsymbols
import sympy as sp
from IPython.display import display, Math
g, t = sp.symbols('g t')
#here mass is uniform, I may as well model with linear quadratic form for mass distribution
M_tar_rb, I_tar_rb_z= sp.symbols('M_targ I_Ztarg')
M_ee_rb, I_ee_rb_z= sp.symbols('M_ee I_Zee')

#NOTE: cool thing symetric matrix means energy is conserved, this is lagrangian so it is
#human_stiffness, blade_stiff = sp.symbols('k_targ k_ee')
#series_spring = human_stiffness + blade_stiff
k_tar_xx, k_tar_xy ,k_tar_yy = sp.symbols('K_XXtarg K_XYtarg K_YYtarg')
k_blade_xx, k_blade_xy ,k_blade_yy = sp.symbols('K_XXfoil K_XYfoil K_YYfoil')

k_tensor_tar = sp.Matrix([[k_tar_xx,k_tar_xy],[k_tar_xy,k_tar_yy]])
k_tensor_ee = sp.Matrix([[k_blade_xx,k_blade_xy],[k_blade_xy,k_blade_yy]])

m_tar_xx, m_tar_xy ,m_tar_yy = sp.symbols('M_XXtarg M_XYtarg M_YYtarg')
m_blade_xx, m_blade_xy ,m_blade_yy = sp.symbols('M_XXfoil M_XYfoil M_YYfoil')

m_tensor_tar = sp.Matrix([[m_tar_xx,m_tar_xy],[m_tar_xy,m_tar_yy]])
m_tensor_ee = sp.Matrix([[m_blade_xx,m_blade_xy],[m_blade_xy,m_blade_yy]])


x_tar,y_tar,phi_tar= dynamicsymbols('x_targ y_targ phi_targ')
x_ee,y_ee,phi_ee= dynamicsymbols('x_ee y_ee phi_ee')

x_tar_dot = x_tar.diff()
y_tar_dot = y_tar.diff()
phi_tar_dot = phi_tar.diff()

x_tar_ddot = x_tar.diff().diff()
y_tar_ddot = y_tar.diff().diff()
phi_tar_ddot = phi_tar.diff().diff()

x_ee_dot = x_ee.diff()
y_ee_dot = y_ee.diff()
phi_ee_dot = phi_ee.diff()

x_tar_ddot = x_ee.diff().diff()
y_ee_ddot = y_ee.diff().diff()
phi_ee_ddot = phi_ee.diff().diff()


## robotic state
target_state_pos = sp.Matrix([x_tar,y_tar]);
end_effector_state_pos = sp.Matrix([x_ee,y_ee]).diff();

target_state_vel = sp.Matrix([x_tar,y_tar]);
end_effector_state_vel = sp.Matrix([x_ee,y_ee]).diff();

display(sp.Matrix.hstack(target_state_pos,end_effector_state_pos,target_state_vel,end_effector_state_vel))




Matrix([
[x_targ(t), Derivative(x_ee(t), t), x_targ(t), Derivative(x_ee(t), t)],
[y_targ(t), Derivative(y_ee(t), t), y_targ(t), Derivative(y_ee(t), t)]])

## lagrangian eqn setup, at collision

In [64]:
stiffness_series = k_tensor_tar + k_tensor_ee
delta_spring_compress = end_effector_state_pos - target_state_pos

T =( 0.5  *(end_effector_state_vel.T*m_tensor_ee * end_effector_state_vel)[0] + .5  * (target_state_vel.T * m_tensor_tar*target_state_vel)[0] + .5 *(I_ee_rb_z*phi_ee + .5 *I_tar_rb_z*phi_tar))

V = .5 * (delta_spring_compress.T * stiffness_series  * delta_spring_compress )[0]
display(T-V)

0.5*I_Zee*phi_ee(t) + 0.25*I_Ztarg*phi_targ(t) + 0.5*(M_XXfoil*Derivative(x_ee(t), t) + M_XYfoil*Derivative(y_ee(t), t))*Derivative(x_ee(t), t) + 0.5*(M_XXtarg*x_targ(t) + M_XYtarg*y_targ(t))*x_targ(t) + 0.5*(M_XYfoil*Derivative(x_ee(t), t) + M_YYfoil*Derivative(y_ee(t), t))*Derivative(y_ee(t), t) + 0.5*(M_XYtarg*x_targ(t) + M_YYtarg*y_targ(t))*y_targ(t) - 0.5*((K_XXfoil + K_XXtarg)*(-x_targ(t) + Derivative(x_ee(t), t)) + (K_XYfoil + K_XYtarg)*(-y_targ(t) + Derivative(y_ee(t), t)))*(-x_targ(t) + Derivative(x_ee(t), t)) - 0.5*((K_XYfoil + K_XYtarg)*(-x_targ(t) + Derivative(x_ee(t), t)) + (K_YYfoil + K_YYtarg)*(-y_targ(t) + Derivative(y_ee(t), t)))*(-y_targ(t) + Derivative(y_ee(t), t))